# Arquitetura do Modelo 6

## Etapa 1 - Importando as bibliotecas

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
import zipfile

cv2.__version__  # confere a versão do opencv instalada

In [ ]:
%tensorflow_version 2.x  # garante que o Colab use o TF 2.x
import tensorflow
tensorflow.__version__  # confere a versão instalada

## Etapa 2 - Conectando com o Drive e acessando os arquivos

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# extrai o material do curso (já baixado do drive na célula anterior)
path = "/content/gdrive/My Drive/Material.zip"
zip_object = zipfile.ZipFile(file=path, mode="r")
zip_object.extractall("./")

# extrai também o dataset fer2013, que fica zipado dentro do Material
base_imgs = 'Material/fer2013.zip'
zip_object = zipfile.ZipFile(file = base_imgs, mode = "r")
zip_object.extractall("./")
zip_object.close()

## Etapa 3 - Acessando a base com fotos de expressões faciais



In [ ]:
# diretório do drive onde estão os arquivos do curso (csv, modelos, fotos de teste etc)
diretorio = 'gdrive/My Drive/Cursos/Deteccao_Expressoes_Faciais/'

data = pd.read_csv(diretorio + 'fer2013/fer2013.csv')
data.tail() # só pra ver as últimas linhas e conferir o formato

In [ ]:
# histograma pra ver como as emoções estão distribuídas no dataset (se tá balanceado ou não)
plt.figure(figsize=(12,6))
plt.hist(data['emotion'], bins=6)
plt.title("Imagens x emoção")
plt.show()

# Classes: ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

## Etapa 4 - Pré-processamento

In [ ]:
# a coluna 'pixels' vem como string, com os valores separados por espaço (ex: "70 80 82 ...")
pixels = data['pixels'].tolist()
largura, altura = 48, 48

faces = [] # vai guardar todas as imagens já convertidas em matriz
amostras = 0
for pixel_sequence in pixels:
  # separa a string de pixels e converte cada valor pra inteiro
  face = [int(pixel) for pixel in pixel_sequence.split(' ')]
  # transforma a lista de pixels numa matriz 48x48 (formato de imagem)
  face = np.asarray(face).reshape(largura, altura)
  faces.append(face)

  if (amostras < 10): # mostra só as 10 primeiras, pra não travar o notebook
    cv2_imshow(face)

  amostras = amostras + 1

faces = np.asarray(faces)
# adiciona uma dimensão extra pro canal de cor (imagem em escala de cinza = 1 canal)
faces = np.expand_dims(faces, -1)

# normaliza os pixels: de 0-255 pra 0-1 (ajuda a rede a convergir mais rápido)
def normalizar(x):
    x = x.astype('float32')
    x = x / 255.0
    return x

faces = normalizar(faces)

# transforma o id da emoção (0 a 6) em one-hot encoding, formato que a rede espera na saída
emocoes = pd.get_dummies(data['emotion']).as_matrix()

In [ ]:
# confere quantas imagens no total foram carregadas
print("Número total de imagens no dataset: "+str(len(faces)))

## Etapa 5 - Imports do Tensorflow/Keras

In [ ]:
from sklearn.model_selection import train_test_split # divide os dados em treino/teste/validação

# camadas e utilitários do keras. aqui usamos "Model" (API funcional) além do Sequential,
# porque a arquitetura Inception precisa ramificar e depois juntar (concatenate) várias camadas,
# o que não dá pra fazer com o Sequential (que só empilha camadas em sequência)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Input, concatenate
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# callbacks: controlam o treinamento (reduzir learning rate, parar cedo, salvar o melhor modelo)
from tensorflow.keras.callbacks import ReduceLROnPlateau, TensorBoard, EarlyStopping, ModelCheckpoint

from tensorflow.keras.models import load_model # carregar um modelo já treinado (.h5)
from tensorflow.keras.models import model_from_json#recriar a arquitetura a partir do json salvo
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # gerador de data augmentation

## Etapa 6 - Dividir em conjuntos para treinamento e validação

In [ ]:
# separa 10% dos dados pra teste
x_train, x_test, y_train, y_test = train_test_split(faces, emocoes, test_size=0.1, random_state=42)
# do que sobrou, separa mais 10% pra validação (usada durante o treinamento)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=41)

print("Número de imagens no conjunto de treinamento:", len(x_train))
print("Número de imagens no conjunto de testes:", len(x_test))
print("Número de imagens no conjunto de validação:", len(y_val))

In [ ]:
# salva o conjunto de teste em disco, pra depois recarregar sem refazer o pré-processamento
np.save('mod_xtest', x_test)
np.save('mod_ytest', y_test)

## Etapa 7 - Arquitetura do Modelo (CNN)

### Arquitetura do modelo - Inception
Arquitetura: https://ieeexplore.ieee.org/document/7477450

In [ ]:
# arquitetura 6: inspirada no Inception (GoogLeNet) - ao invés de empilhar convoluções em
# sequência, cada "módulo" aplica VÁRIAS convoluções de tamanhos diferentes (1x1, 3x3, 5x5) em
# paralelo sobre a mesma entrada, e depois concatena (junta) todos os resultados. a ideia é deixar
# a rede "escolher" qual tamanho de filtro funciona melhor pra cada característica, ao invés de
# a gente decidir isso na mão
num_classes = 7
width, height = 48, 48
batch_size = 128
epochs = 100

model = Sequential()  # não é usado de fato (a arquitetura é montada com a API funcional abaixo)

input_img = Input(shape=(width, height, 1))  # camada de entrada

# bloco convolucional inicial (reduz a imagem de 48x48 pra um mapa de características menor)
layer1 = Conv2D(10, (3, 3), padding='same', activation='relu')(input_img)
layer1_2 = Conv2D(20, (3, 3), padding='same', activation='relu')(layer1)
layer2 = MaxPooling2D(pool_size=(3, 3))(layer1_2)
bn1 = BatchNormalization()(layer2)

layer3 = Conv2D(30, (3, 3), padding='same', activation='relu')(bn1)
layer3_2 = Conv2D(40, (3, 3), padding='same', activation='relu')(layer3)
layer4 = MaxPooling2D(pool_size=(3, 3))(layer3_2)
bn2 = BatchNormalization()(layer4)

layer5 = Conv2D(50, (3, 3), padding='same', activation='relu')(bn2)
layer5_2 = Conv2D(60, (3, 3), padding='same', activation='relu')(layer5)
layer6 = MaxPooling2D(pool_size=(3, 3))(layer5_2)
bn3 = BatchNormalization()(layer6)

# --- primeiro módulo "inception": 3 convoluções em paralelo (1x1, 3x3, 5x5) sobre o bn3 ---
Conv11 = Conv2D(1, (1, 1), padding='same', activation='relu')(bn3)
Conv33 = Conv2D(4, (3, 3), padding='same', activation='relu')(bn3)
Conv332 = Conv2D(1, (3, 3), padding='same', activation='relu')(Conv33)
Conv55 = Conv2D(4, (5, 5), padding='same', activation='relu')(bn3)
Conv552 = Conv2D(1, (3, 3), padding='same', activation='relu')(Conv55)
#Pool33 = MaxPooling2D(pool_size=(3, 3))(bn3)
#ConvPool1 = Conv2D(4, (1, 1), padding='same', activation='relu')(Pool33)

# junta (concatena) as 3 saídas em paralelo num único bloco de saída
intermediate1 = concatenate([Conv11, Conv332, Conv552], axis=1)

# --- segundo módulo "inception", igual ao de cima mas em cima da saída do primeiro ---
Conv2_11 = Conv2D(1, (1, 1), padding='same', activation='relu')(intermediate1)
Conv2_33 = Conv2D(4, (3, 3), padding='same', activation='relu')(intermediate1)
Conv2_332 = Conv2D(1, (3, 3), padding='same', activation='relu')(Conv2_33)
Conv2_55 = Conv2D(4, (5, 5), padding='same', activation='relu')(intermediate1)
Conv2_552 = Conv2D(1, (3, 3), padding='same', activation='relu')(Conv2_55)
#Pool2_33 = MaxPooling2D(pool_size=(3, 3))(intermediate1)
#ConvPool2 = Conv2D(4, (1, 1), padding='same', activation='relu')(Pool2_33)

intermediate2 = concatenate([Conv2_11, Conv2_332, Conv2_552], axis=1)

# terceiro módulo "inception"
Conv3_11 = Conv2D(1, (1, 1), padding='same', activation='relu')(intermediate2)
Conv3_33 = Conv2D(4, (3, 3), padding='same', activation='relu')(intermediate2)
Conv3_332 = Conv2D(1, (3, 3), padding='same', activation='relu')(Conv3_33)
Conv3_55 = Conv2D(4, (5, 5), padding='same', activation='relu')(intermediate2)
Conv3_552 = Conv2D(1, (3, 3), padding='same', activation='relu')(Conv3_55)
#Pool3_33 = MaxPooling2D(pool_size=(3, 3))(intermediate2)
#ConvPool3 = Conv2D(4, (1, 1), padding='same', activation='relu')(Pool3_33)

intermediate3 = concatenate([Conv3_11, Conv3_332, Conv3_552], axis=1)

#Pool4 = MaxPooling2D(pool_size=(3, 3))(intermediate3)

Flat = Flatten()(intermediate3) # achata pra entrar nas camadas densas

# classificador final (bem pequeno nesse exemplo)
Dense1 = Dense(25, activation='relu')(Flat)
Dense2 = Dense(15, activation='relu')(Dense1)
Dense3 = Dense(num_classes, activation='softmax')(Dense2)  # camada de saída: 7 emoções

# monta o modelo funcional a partir da entrada e da saída definidas acima
model = Model([input_img], Dense3)

print(model.summary())

In [ ]:
# gerador de data augmentation (rotaciona, distorce, dá zoom, desloca, espelha as imagens)
datagen = ImageDataGenerator(
      rotation_range=10,
      shear_range=0.1,
      zoom_range=0.1,
      width_shift_range=0.1,
      height_shift_range=0.1,
      horizontal_flip=True,
      fill_mode='nearest')

# essa linha sobrescreve o datagen acima por um sem nenhuma transformação
# (ou seja, nesse notebook o data augmentation configurado acima acaba não sendo usado de fato)
datagen = ImageDataGenerator()

## Etapa 8 - Compilando o modelo

In [ ]:
model.compile(loss=categorical_crossentropy, # loss padrão pra classificação com várias classes
              optimizer=Adam(lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-7),
              metrics=['accuracy'])
arquivo_modelo = diretorio + "modelo_06_expressoes.h5" # arquivo do modelo
arquivo_modelo_json = diretorio + "modelo_06_expressoes.json" # arquivo do json, para salvar a arquitetura
# reduz o learning rate quando a loss de validação para de melhorar
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.9, patience=3, verbose=1)
# para o treinamento mais cedo se não houver melhora (evita overfitting)
early_stopper = EarlyStopping(monitor='val_loss', min_delta=0, patience=8, verbose=1, mode='auto')
# salva automaticamente o melhor modelo (menor val_loss) durante o treino
checkpointer = ModelCheckpoint(arquivo_modelo, monitor='val_loss', verbose=1, save_best_only=True)

### Salvando a arquitetura do modelo em um arquivo JSON

In [ ]:
# salva só a arquitetura do modelo (as camadas) num arquivo json
# os pesos ficam separados, no .h5 salvo pelo checkpointer durante o treino
model_json = model.to_json()
with open(arquivo_modelo_json, "w") as json_file:
    json_file.write(model_json)

## Etapa 9 - Treinando o modelo

In [ ]:
# usamos fit_generator() ao invés de fit() porque os dados de treinamento vêm de um gerador (datagen)
history = model.fit_generator(
          datagen.flow(x_train, y_train, batch_size=batch_size),
          epochs=epochs,
          verbose=1,
          validation_data= (x_val, y_val),
          validation_steps = len(x_val) // batch_size,
          steps_per_epoch = len(x_train) // batch_size,
          callbacks=[lr_reducer, early_stopper, checkpointer])

Outras arquiteturas e modelos:
 * Xception - https://arxiv.org/abs/1610.02357
 * DeXpression - https://arxiv.org/abs/1509.05371
 * Mais versões do Inception: https://towardsdatascience.com/a-simple-guide-to-the-versions-of-the-inception-network-7fc52b863202 / https://maelfabien.github.io/deeplearning/inception/

## Gerando gráfico da melhora em cada etapa do treinamento

In [ ]:
# plota dois gráficos lado a lado: acurácia e loss (treino x validação) ao longo das epochs
def plota_historico_modelo(historico_modelo):
    fig, axs = plt.subplots(1,2,figsize=(15,5))
    axs[0].plot(range(1,len(historico_modelo.history['accuracy'])+1),
                historico_modelo.history['accuracy'],'r')
    axs[0].plot(range(1,len(historico_modelo.history['val_accuracy'])+1),
                historico_modelo.history['val_accuracy'],'b')
    axs[0].set_title('Acurácia do Modelo')
    axs[0].set_ylabel('Acuracia')
    axs[0].set_xlabel('Epoch')
    axs[0].set_xticks(np.arange(1,len(historico_modelo.history['accuracy'])+1),
                      len(historico_modelo.history['accuracy'])/10)
    axs[0].legend(['training accuracy', 'validation accuracy'], loc='best')

    axs[1].plot(range(1,len(historico_modelo.history['loss'])+1),
                historico_modelo.history['loss'],'r')
    axs[1].plot(range(1,len(historico_modelo.history['val_loss'])+1),
                historico_modelo.history['val_loss'],'b')
    axs[1].set_title('Perda/Loss do Modelo')
    axs[1].set_ylabel('Loss')
    axs[1].set_xlabel('Epoch')
    axs[1].set_xticks(np.arange(1,len(historico_modelo.history['loss'])+1),
                      len(historico_modelo.history['loss'])/10)
    axs[1].legend(['training loss', 'validation Loss'], loc='best')
    fig.savefig('historico_modelo_mod06.png')  # salva a imagem do gráfico em disco
    plt.show()

plota_historico_modelo(history)

### Verificando a acurácia do modelo

In [ ]:
# avalia o modelo já treinado usando o conjunto de teste (imagens que ele nunca viu)
scores = model.evaluate(np.array(x_test), np.array(y_test), batch_size=batch_size)
print("Acurácia: " + str(scores[1]))
print("Perda/Loss: " + str(scores[0]))

## Carregaremos os dados para gerar a matriz de confusão

In [ ]:
true_y=[]  # vai guardar o índice da emoção correta (rótulo verdadeiro)
pred_y=[]  # vai guardar o índice da emoção que o modelo previu

# recarrega o conjunto de teste que salvamos antes
x = np.load('mod_xtest.npy')
y = np.load('mod_ytest.npy')

# recarrega o modelo a partir da arquitetura (json) + pesos (h5)
json_file = open(arquivo_modelo_json, 'r')
loaded_model_json = json_file.read()
json_file.close()
loaded_model = model_from_json(loaded_model_json)
loaded_model.load_weights(arquivo_modelo)

y_pred= loaded_model.predict(x)  # faz a previsão pra todo o conjunto de teste

yp = y_pred.tolist()
yt = y.tolist()
count = 0
for i in range(len(y)):
    yy = max(yp[i]) # maior probabilidade prevista
    yyt = max(yt[i]) # valor "1" do one-hot (rótulo verdadeiro)
    pred_y.append(yp[i].index(yy)) # índice da emoção prevista
    true_y.append(yt[i].index(yyt)) # índice da emoção correta
    if(yp[i].index(yy)== yt[i].index(yyt)):
        count+=1 # conta quantas vezes acertou
acc = (count/len(y))*100

# salva os resultados (usados na matriz de confusão)
np.save('truey__mod06', true_y)
np.save('predy__mod06', pred_y)
print("Acurácia no conjunto de testes: "+str(acc)+"%")

## Gerando a Matriz de Confusão

In [ ]:
from sklearn.metrics import confusion_matrix

y_true = np.load('truey__mod06.npy')
y_pred = np.load('predy__mod06.npy')

cm = confusion_matrix(y_true, y_pred)  # monta a matriz comparando o rótulo real x o previsto
expressoes = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]
titulo='Matriz de Confusão'
print(cm)

In [ ]:
# desenha a matriz de confusão como um "mapa de calor"
import itertools
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title(titulo)
plt.colorbar()
tick_marks = np.arange(len(expressoes))
plt.xticks(tick_marks, expressoes, rotation=45)
plt.yticks(tick_marks, expressoes)
fmt = 'd'
thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    # escreve o número dentro de cada célula da matriz
    plt.text(j, i, format(cm[i, j], fmt),
            horizontalalignment="center",
            color="white" if cm[i, j] > thresh else "black")

plt.ylabel('Classificação Correta')
plt.xlabel('Predição')
plt.savefig('matriz_confusao_mod06.png')
plt.show()

## Testando brevemente o modelo

In [ ]:
imagem = cv2.imread(diretorio + "testes/teste02.jpg")
cv2_imshow(imagem)

In [ ]:
# recarrega o melhor checkpoint salvo durante o treino, pra rodar o teste rápido abaixo
model = load_model(diretorio + "modelo_06_expressoes.h5")
scores = model.evaluate(np.array(x_test), np.array(y_test), batch_size=batch_size)
print("Perda/Loss: " + str(scores[0]))
print("Acurácia: " + str(scores[1]))

In [ ]:
expressoes = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]
original = imagem.copy()
gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
face_cascade = cv2.CascadeClassifier(diretorio + 'haarcascade_frontalface_default.xml')
faces = face_cascade.detectMultiScale(gray, 1.1, 3)
for (x, y, w, h) in faces:
    cv2.rectangle(original, (x, y), (x + w, y + h), (0, 255, 0), 1)  # desenha o retângulo em volta do rosto
    roi_gray = gray[y:y + h, x:x + w] # extrai só a região do rosto (ROI)
    roi_gray = roi_gray.astype("float") / 255.0 # normaliza
    cropped_img = np.expand_dims(np.expand_dims(cv2.resize(roi_gray, (48, 48)), -1), 0)  # redimensiona e ajusta o shape
    cv2.normalize(cropped_img, cropped_img, alpha=0, beta=1,
                  norm_type=cv2.NORM_L2, dtype=cv2.CV_32F)
    prediction = model.predict(cropped_img)[0] # prediz a emoção
    cv2.putText(original, expressoes[int(np.argmax(prediction))], (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)  # escreve a emoção acima do rosto
cv2_imshow(original)